# Task - 1: Table Booking and Online Delivery

## Importing required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import scipy.stats as st
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None) 
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.float_format', '{:.3f}'.format) 

sns.set_style("whitegrid")

## Loading Data

In [ ]:
base_path = Path.cwd().parents[1]  
file_path = base_path / "Notebooks" / "processed_data" / "Dataset_filtered.csv"
df = pd.read_csv(file_path)

## Initial inspection

In [ ]:
df.head(10)

In [ ]:
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

In [ ]:
df.info()

In [ ]:
# missing value inspection
df.isnull().sum()

In [ ]:
# duplicate rows inspection
df.duplicated(keep=False).any()

In [ ]:
df.columns

In [ ]:
df_1 = df[['Restaurant ID', 'Restaurant Name','Has Table booking','Has Online delivery', 
            'Is delivering now', 'Switch to order menu','Price range', 'Aggregate rating', 'Rating text', 'Votes']]

df_1.head()

In [ ]:
mapping = {1:'Budget' ,2:'Mid' ,3:'Premium' ,4:'Luxury'}
df_1['Price range'] = df_1['Price range'].replace(mapping)
df_1.head(10)

### **Table Booking and Online Delivery**

In [ ]:
for col in ['Has Table booking','Has Online delivery']:
    print(f'___________________{col}___________________')
    print(df_1[col].value_counts(normalize=True)*100)
    print('='*40)
    print('\n')

In [ ]:
table_pct   = df_1['Has Table booking'].value_counts(normalize=True) * 100
delivery_pct = df_1['Has Online delivery'].value_counts(normalize=True) * 100

categories = ['Table Booking', 'Online Delivery']
yes_vals   = [table_pct.get('Yes', 0), delivery_pct.get('Yes', 0)]
no_vals    = [table_pct.get('No', 0),  delivery_pct.get('No', 0)]

# Create figure and axis
fig, ax = plt.subplots()

fig.suptitle('Table Booking & Online Delivery Analysis', fontsize=15, fontweight='bold')

x = np.arange(len(categories))

ax.bar(x - 0.2, yes_vals, width=0.4, label='Yes', color='#1D9E75', edgecolor='white')
ax.bar(x + 0.2, no_vals,  width=0.4, label='No',  color='#E24B4A', edgecolor='white')

# Labels on bars
for i, (y, n) in enumerate(zip(yes_vals, no_vals)):
    ax.text(i - 0.2, y + 0.5, f'{y:.1f}%', ha='center', fontsize=9, fontweight='bold')
    ax.text(i + 0.2, n + 0.5, f'{n:.1f}%', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel('Percentage (%)')
ax.set_title('% Restaurants with\nTable Booking & Online Delivery')
plt.tight_layout(rect=[0, 0, 1, 0.92])
ax.legend()
ax.set_ylim(0, 115)

base_path = Path.cwd().parents[1]
file_path = base_path / "Notebooks" / "reports" / "Table_Booking_Online_Delivery_Analysis.png"
plt.savefig(file_path, dpi=150, bbox_inches='tight')
plt.show()

**Observation:**
- Only ~12% restaurants ***have Table Booking***.
- Only ~26% restaurants ***have Online Delivery***.

### **Average rating with vs without Table Booking**

In [ ]:
df_1

In [ ]:
df_1[['Aggregate rating']].describe()

In [ ]:
temp = df_1[df_1['Aggregate rating']==0.0][['Aggregate rating', 'Rating text', 'Votes']]
temp

In [ ]:
print(temp['Rating text'].value_counts())
print()
print(temp['Votes'].value_counts())

**Note:**
- From previous analysis, we have seen that
   * Exactly 2,148 restaurants (22.5%) have a rating of 0.0, and the Rating text column labels them "Not rated" — these are definitely missing values, not true zero ratings.
   * Crucially, 1,054 out of 2148 of these "Not rated" entries actually have non-zero votes, meaning some users voted but the platform never computed an aggregate. This makes them a distinct data quality issue.
- For further analysis, we are ignoring those restaurants which have a aggregate rating of 0.0

In [ ]:
# Average rating of restaurants with vs without table booking

rated = df_1[df_1['Aggregate rating'] > 0]
booking_rating = rated.groupby('Has Table booking')[['Aggregate rating']].mean()
booking_rating

In [ ]:
rated = df_1[df_1['Aggregate rating'] > 0]
booking_rating = rated.groupby('Has Table booking')['Aggregate rating'].mean()

fig, ax = plt.subplots()
colors = ['#1D9E75' if i == 'Yes' else '#E24B4A' for i in booking_rating.index]
bars = ax.bar(booking_rating.index, booking_rating.values,
                   color=colors, edgecolor='white', width=0.5)

for bar, val in zip(bars, booking_rating.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02,
                 f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Has Table Booking')
ax.set_ylabel('Average Rating')
ax.set_title('Avg Rating: With vs Without\nTable Booking')
ax.set_ylim(0, 5)
ax.axhline(y=rated['Aggregate rating'].mean(), color='gray', linestyle='--',
                linewidth=1, label=f"Overall avg ({rated['Aggregate rating'].mean():.2f})")
ax.legend(fontsize=9)

base_path = Path.cwd().parents[1]
file_path = base_path / "Notebooks" / "reports" / "Avg_Rating_With_vs_Without_Table_Booking.png"
plt.savefig(file_path, dpi=150, bbox_inches='tight')
plt.show()

**Observation:**
- The average rating is slightly higher for those restaurants which ***have table booking facility***.

### **Online Delivery and Price Range**

In [ ]:
df_1['Price range'].value_counts()

In [ ]:
cats = list(df_1['Price range'].value_counts().index)
cats

In [ ]:
for cat in cats:
    print(f"_______________{cat}_______________")
    print(df_1[df_1['Price range']==cat]['Has Online delivery'].value_counts(normalize=True)*100)
    print("="*35)
    print("\n")

In [ ]:
fig, ax = plt.subplots()
price_delivery = (df_1.groupby(['Price range', 'Has Online delivery'])
                    .size()
                    .unstack(fill_value=0))

order = ['Budget', 'Mid', 'Premium', 'Luxury']
price_delivery = price_delivery.reindex(order)
price_delivery_pct = price_delivery.div(price_delivery.sum(axis=1), axis=0) * 100

x = np.arange(len(price_delivery_pct))

ax.bar(x, price_delivery_pct.get('Yes', 0), label='Delivery: Yes',
            color='#1D9E75', edgecolor='white')
ax.bar(x, price_delivery_pct.get('No', 0),  label='Delivery: No',
            color='#E24B4A', edgecolor='white',
            bottom=price_delivery_pct.get('Yes', 0))

for i, (yes_v, no_v) in enumerate(zip(price_delivery_pct.get('Yes', [0]*4),
                                       price_delivery_pct.get('No',  [0]*4))):
    ax.text(i, yes_v / 2,        f'{yes_v:.1f}%', ha='center', fontsize=9,
                 fontweight='bold', color='white')
    ax.text(i, yes_v + no_v / 2, f'{no_v:.1f}%',  ha='center', fontsize=9,
                 fontweight='bold', color='white')

ax.set_xticks(x)
ax.set_xticklabels([i for i in price_delivery_pct.index], fontsize=9)
ax.set_ylabel('Percentage (%)')
ax.set_title('Online Delivery Availability\nby Price Range')
ax.legend()
ax.set_ylim(0, 110)
base_path = Path.cwd().parents[1]
file_path = base_path / "Notebooks" / "reports" / "Online_Delivery_Availability_by_Price_Range.png"
plt.savefig(file_path, dpi=150, bbox_inches='tight')
plt.show()

# Task - 2: Price Range Analysis

In [ ]:
df_1.head()

In [ ]:
df_1['Price range'].value_counts(normalize=True)*100

In [ ]:
values = list(df_1['Price range'].value_counts(normalize=True).values)
values = [val*100 for val in values]
index = list(df_1['Price range'].value_counts(normalize=True).index)

colors = ["#0D960D", "#DADA1B", "#FFA500", "#D30A0A"]
fig, ax = plt.subplots()

bars = ax.bar(index, values, color=colors, edgecolor='white', width=0.5)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02,
                 f'{val:.2f}%', ha='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Price Range')
ax.set_ylabel('Available % among all restaurants')
ax.set_title('Common price range analysis')
base_path = Path.cwd().parents[1]
file_path = base_path / "Notebooks" / "reports" / "Common_price_range_analysis.png"
plt.savefig(file_path, dpi=150, bbox_inches='tight')
plt.show()

**Observation:**
- ***Budget Price Range*** is the most common price range among all the restaurants

In [ ]:
# Average rating for each price range

rated = df_1[df_1['Aggregate rating'] > 0]
price_rating = rated.groupby('Price range')[['Aggregate rating']].mean().sort_values(by='Aggregate rating')
price_rating

In [ ]:
values = [round(val, 2) for val in list(price_rating['Aggregate rating'])]
index = list(price_rating.index)

colors = ["#0D960D", "#DADA1B", "#FFA500", "#D30A0A"]
fig, ax = plt.subplots()

bars = ax.bar(index, values, color=colors, edgecolor='white', width=0.5)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02,
                 f'{val:.2f}%', ha='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Price range')
ax.set_ylabel('Average Rating')
ax.set_title('Avg Rating for different price range')

ax.axhline(y=rated['Aggregate rating'].mean(), color='gray', linestyle='--',
                linewidth=1, label=f"Overall avg ({rated['Aggregate rating'].mean():.2f})")
ax.legend(fontsize=9)
base_path = Path.cwd().parents[1]
file_path = base_path / "Notebooks" / "reports" / "Avg_Rating_for_different_price_range.png"
plt.savefig(file_path, dpi=150, bbox_inches='tight')
plt.show()

**Observation:**
- ***Luxury price range*** has better average rating compared to other price ranges

### Identifying the color that represents the highest average rating among different price ranges

In [ ]:
cats

In [ ]:
for cat in cats:
    print(f"__________{cat}__________")
    val = df_1[df_1['Price range']==cat]['Aggregate rating'].max()
    print("Maximum rating: {}".format(val))
    color = df[(df_1['Price range']==cat) & (df_1['Aggregate rating']==val)]['Rating color'].unique()[0]
    print("Related rating color: {}".format(color))
    print("="*40)
    print("\n")

**Observation:**
- Every price range has 4.9 as it's maximum rating and the related rating color is ***Dark Green***

# Task - 3: Feature Engineering

In [ ]:
df.head()

In [ ]:
# calculating length of restaurant name
df.insert(loc=2, column='Restaurant Name Length', value=df['Restaurant Name'].apply(lambda x: len(x)))

In [ ]:
# longest and shortest restaurant name
print(f"Longest restaurant name(s):\n{df[df['Restaurant Name Length']==df['Restaurant Name Length'].max()]['Restaurant Name'].squeeze()}")
print(f"Length: {df['Restaurant Name Length'].max()}")
print()
print(f"Shortest restaurant name(s):\n{df[df['Restaurant Name Length']==df['Restaurant Name Length'].min()]['Restaurant Name'].squeeze()}")
print(f"Length: {df['Restaurant Name Length'].min()}")

In [ ]:
# calculating length of address
df.insert(loc=5, column='Address Length', value=df['Address'].apply(lambda x: len(x)))

In [ ]:
# longest and shortest address
print(f"Longest address(s):\n{df[df['Address Length']==df['Address Length'].max()]['Address'].squeeze()}")
print(f"Length: {df['Address Length'].max()}")
print()
print(f"Shortest address(s):\n{df[df['Address Length']==df['Address Length'].min()]['Address'].squeeze()}")
print(f"Length: {df['Address Length'].min()}")

In [ ]:
df.head()

In [ ]:
df.columns

### Feature Encoding

In [ ]:
df.dtypes

In [ ]:
df.columns

In [ ]:
df_2 = df.drop(labels=['Restaurant Name','Address', 'Locality',
       'Locality Verbose','Switch to order menu'],axis=1)
df_2

In [ ]:
num_cols = [col for col in df_2.columns if df[col].dtypes!='str']
cat_cols = [col for col in df_2.columns if col not in num_cols]
print(num_cols,'\n',cat_cols)

In [ ]:
# Binary Encoding
mapping = {'No':0, 'Yes':1}

binary_columns = ['Has Table booking', 'Has Online delivery', 'Is delivering now']

for col in binary_columns:
    df_2[col] = df_2[col].replace(mapping)

df_2.head()

In [ ]:
# Ordinal Encoding
rating_text_order = ['Not rated', 'Poor', 'Average', 'Good', 'Very Good', 'Excellent']
df_2['Rating text'] = df_2['Rating text'].map({v: i for i, v in enumerate(rating_text_order)})
 
rating_color_order = ['White', 'Red', 'Orange', 'Yellow', 'Green', 'Dark Green']
df_2['Rating color'] = df_2['Rating color'].map({v: i for i, v in enumerate(rating_color_order)})

df_2.head()

In [ ]:
# Label Encoder
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_2['Currency'] = le.fit_transform(df_2['Currency'])
df_2['City']     = le.fit_transform(df_2['City'])

df_2.head()

In [ ]:
# Multi Label Binarizer
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()
cuisines_split  = df['Cuisines'].str.split(', ')
cuisine_matrix  = mlb.fit_transform(cuisines_split)
cuisine_df      = pd.DataFrame(cuisine_matrix,
                               columns=[f'Cuisine_{c}' for c in mlb.classes_])
print(f"Cuisine size: {cuisine_df.shape}")
cuisine_df.head()

In [ ]:
df_2 = df_2.drop('Cuisines', axis=1)

df_2 = pd.concat([df_2, cuisine_df], axis=1)

df_2.head()